# 02 Data reading

Reads the 156 Datastream block exports (monthly, weekly and annual) and the two standalone series (EURIBOR and market indices), and stacks them into long-format tables saved as Parquet. Section 3.4.1.

Outputs: `raw_monthly`, `raw_weekly`, `raw_annual`, `fiscal_year_end`, `security_meta`, `raw_riskfree`, `raw_index` (Parquet) and `read_log.csv`.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, glob
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

files = sorted(glob.glob(os.path.join(DATA_DIR, "MKT_*.xlsx")))
print(f"Block files found: {len(files)}")
for fam, label in [('M', 'monthly'), ('W', 'weekly'), ('A', 'annual')]:
    n = len([f for f in files if f"_{fam}_" in os.path.basename(f)])
    print(f"  {label:9s}: {n:>3}")

extra = sorted(glob.glob(os.path.join(DATA_DIR, "RF_*.xlsx")) +
               glob.glob(os.path.join(DATA_DIR, "IDX_*.xlsx")))
print(f"\nStandalone series: {[os.path.basename(f) for f in extra] or 'none found'}")

## Parser

`parse_ds_file` locates the `Code` header row, splits each column heading into a
`SYMBOL(DATATYPE)` pair, and stacks the values into long format.
`aggregate_meta` collapses the per-column metadata to one row per security.

In [ ]:
import openpyxl, re, os, glob, time
import pandas as pd
import numpy as np

# Datatypes holding dates rather than numeric values
DATE_DATATYPES = {'WC05350'}

# Patterns for extracting the delisting date from the series name
DELIST_PATS = [
    r'DELIST\.?\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'\bDEL\.?\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'DEAD\s*-\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'MERGER\s*(\d{1,2}/\d{1,2}/\d{2,4})',
    r'(\d{1,2}/\d{1,2}/\d{2,4})',
]


def parse_delist_date(s):
    """'28/05/04' -> Timestamp. Two-digit years pivot at 1970."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return pd.NaT
    try:
        d, m, y = str(s).split('/')
        y = int(y)
        if y < 100:
            y = 2000 + y if y < 70 else 1900 + y
        return pd.Timestamp(year=y, month=int(m), day=int(d))
    except Exception:
        return pd.NaT


def _extract_meta_row(sym, name):
    up = name.upper()
    is_dead = 'DEAD' in up
    dd = None
    if is_dead or 'MERGER' in up or 'DELIST' in up:
        for p in DELIST_PATS:
            mm = re.search(p, name)
            if mm:
                dd = mm.group(1)
                break
    return {'symbol': sym, 'name_raw': name, 'is_dead': is_dead, 'delist_str': dd}


def parse_ds_file(path):
    """Parse one Datastream export saved with 'Display Code' enabled.
    Returns numeric values and dates in long format, per-column metadata and the number of #ERROR columns."""
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb[wb.sheetnames[0]]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()

    ci = next((i for i, r in enumerate(rows[:10]) if r and r[0] == 'Code'), None)
    if ci is None:
        raise ValueError(f"'Code' row not found in {os.path.basename(path)}")

    code_row, name_row, data_rows = rows[ci], rows[ci - 1], rows[ci + 1:]
    dates = [r[0] for r in data_rows if r and r[0] is not None]
    n = len(dates)

    colmap, n_err = {}, 0
    for j, cd in enumerate(code_row):
        if j == 0:
            continue
        if cd is None:
            if j < len(name_row) and name_row[j] and '#ERROR' in str(name_row[j]):
                n_err += 1
            continue
        m = re.match(r'^(.+?)\(([^)]+)\)$', str(cd))
        if m:
            colmap[j] = (m.group(1).strip(), m.group(2).strip())
        else:
            colmap[j] = (str(cd).strip(), 'X')

    num_parts, date_parts, meta_rows = [], [], []
    for j, (sym, dt) in colmap.items():
        vals = [r[j] if j < len(r) else None for r in data_rows[:n]]

        if dt in DATE_DATATYPES:
            s = pd.Series(vals)
            s = s.where(s.apply(lambda x: hasattr(x, 'year')))
            if s.notna().sum() > 0:
                date_parts.append(pd.DataFrame({'symbol': sym, 'date': dates,
                                                'datatype': dt, 'value_date': s.values}))
        else:
            s = pd.to_numeric(pd.Series(vals).replace('NA', np.nan), errors='coerce')
            if s.notna().sum() > 0:
                num_parts.append(pd.DataFrame({'symbol': sym, 'date': dates,
                                               'datatype': dt, 'value': s.values}))

        nm = str(name_row[j]) if j < len(name_row) and name_row[j] else ''
        meta_rows.append(_extract_meta_row(sym, nm))

    values = (pd.concat(num_parts, ignore_index=True).dropna(subset=['value'])
              if num_parts else pd.DataFrame(columns=['symbol', 'date', 'datatype', 'value']))
    dates_df = (pd.concat(date_parts, ignore_index=True).dropna(subset=['value_date'])
                if date_parts else pd.DataFrame(columns=['symbol', 'date', 'datatype', 'value_date']))
    meta_raw = pd.DataFrame(meta_rows)
    return values, dates_df, meta_raw, n_err


def aggregate_meta(meta_raw):
    """One row per security: dead if any of its columns is marked DEAD, earliest parsed delisting date."""
    meta_raw = meta_raw.copy()
    meta_raw['delist_date'] = meta_raw['delist_str'].apply(parse_delist_date)
    g = meta_raw.groupby('symbol')

    is_dead = g['is_dead'].any()
    dead_names = meta_raw[meta_raw['is_dead']].groupby('symbol')['name_raw'].first()
    any_names = g['name_raw'].first()
    name_raw = dead_names.reindex(any_names.index).fillna(any_names)
    delist_date = g['delist_date'].min()

    return pd.DataFrame({'name_raw': name_raw,
                         'is_dead': is_dead,
                         'delist_date': delist_date}).reset_index()


def load_all(data_dir, pattern='MKT_*.xlsx', verbose=True):
    """Read every block file and consolidate by frequency family."""
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    fam_vals = {'M': [], 'W': [], 'A': []}
    fam_dates, metas, log = [], [], []

    for f in files:
        base = os.path.basename(f)
        parts = base.replace('.xlsx', '').split('_')
        if len(parts) < 3 or parts[1] not in fam_vals:
            print(f"  skipped, unrecognised filename: {base}")
            continue
        fam, blk = parts[1], parts[2]

        t = time.time()
        try:
            v, d, md, nerr = parse_ds_file(f)
        except Exception as e:
            print(f"  could not read {base}: {e}")
            log.append({'file': base, 'freq': fam, 'block': blk, 'rows': 0, 'symbols': 0,
                        'err_cols': None, 'sec': 0, 'status': f'FAIL: {e}'})
            continue

        v['block'] = blk
        fam_vals[fam].append(v)
        if len(d):
            d['block'] = blk
            fam_dates.append(d)
        metas.append(md)

        log.append({'file': base, 'freq': fam, 'block': blk, 'rows': len(v),
                    'symbols': v['symbol'].nunique(), 'err_cols': nerr,
                    'sec': round(time.time() - t, 1), 'status': 'OK'})
        if verbose:
            print(f"  {base:24s} {len(v):>10,} rows  {v['symbol'].nunique():>4} sec  "
                  f"{nerr:>5} #ERR  {time.time()-t:>5.1f}s")

    out = {fam: (pd.concat(fam_vals[fam], ignore_index=True) if fam_vals[fam] else pd.DataFrame())
           for fam in ['M', 'W', 'A']}
    fy = pd.concat(fam_dates, ignore_index=True) if fam_dates else pd.DataFrame()
    meta = aggregate_meta(pd.concat(metas, ignore_index=True)) if metas else pd.DataFrame()

    return out, fy, meta, pd.DataFrame(log)


## Read all blocks


In [ ]:
out, fy, meta, log = load_all(DATA_DIR, pattern="MKT_*.xlsx", verbose=True)

## Summary

In [ ]:
print("Volumes:")
for fam, label in [('M', 'monthly'), ('W', 'weekly'), ('A', 'annual')]:
    d = out[fam]
    if len(d):
        print(f"{label:9s}: {len(d):>12,} rows | {d['symbol'].nunique():>6,} securities | "
              f"{d['block'].nunique():>3} blocks")
        print(f"           datatypes: {sorted(d['datatype'].unique())}")

print(f"\nFiscal year ends : {len(fy):>12,} rows | "
      f"{fy['symbol'].nunique() if len(fy) else 0:>6,} securities")
print(f"Security metadata: {len(meta):>12,} securities | "
      f"{int(meta['is_dead'].sum()):,} dead | "
      f"{int(meta['delist_date'].notna().sum()):,} with delisting date")

print("\nFiles that could not be read:")
bad = log[log['status'] != 'OK']
print(bad.to_string(index=False) if len(bad) else "  none")

print("\nBlocks with most #ERROR columns:")
print(log.nlargest(10, 'err_cols')[['file', 'symbols', 'err_cols']].to_string(index=False))

### Coverage by datatype

Share of securities with at least one valid observation. Sparse datatypes translate
directly into sparse characteristics downstream.

In [ ]:
for fam, label in [('M', 'MONTHLY'), ('A', 'ANNUAL')]:
    d = out[fam]
    if not len(d):
        continue
    print(f"{label}:")
    cov = d.groupby('datatype')['symbol'].nunique().sort_values(ascending=False)
    tot = d['symbol'].nunique()
    for dt, n in cov.items():
        print(f"  {dt:10s} {n:>6,} / {tot:,}  ({100*n/tot:>5.1f}%)")
    print()

### Cross-check against the master universe

Compares the dead/active status inferred from the series names with the status
recorded in the Navigator extraction.

In [ ]:
mp = os.path.join(DATA_DIR, "master_universe.csv")
if os.path.exists(mp):
    u = pd.read_csv(mp, dtype=str)
    m = meta.merge(u[['Symbol', 'Activity', 'country']], left_on='symbol',
                   right_on='Symbol', how='left')
    print("Navigator status (rows) vs 'DEAD' marker in series name (columns):")
    print(pd.crosstab(m['Activity'], m['is_dead'], dropna=False))

    n_dead = int((m['Activity'] == 'Dead').sum())
    n_nodate = int(((m['Activity'] == 'Dead') & m['delist_date'].isna()).sum())
    print(f"\nDelisted securities without a parsed date: {n_nodate:,} of {n_dead:,} "
          f"({100*n_nodate/max(n_dead,1):.1f}%)")
    print("For these the delisting point will be recovered by truncating the return index.")

    univ = set(u['Symbol'])
    have = set(meta['symbol'])
    print(f"\nUniverse: {len(univ):,} | present in the data: {len(have & univ):,} "
          f"({100*len(have & univ)/len(univ):.1f}%)")
    missing = univ - have
    if missing:
        d = u[u['Symbol'].isin(missing)]
        print(f"Never retrieved: {len(missing):,}  by country {d['country'].value_counts().to_dict()}")
else:
    print("master_universe.csv not found, cross-check skipped.")

## Save

In [ ]:
def save(df, name):
    if not len(df):
        print(f"  {name} is empty, not written")
        return
    p = os.path.join(DATA_DIR, name)
    df.to_parquet(p, index=False)
    print(f"  {name:26s} {len(df):>12,} rows  ({os.path.getsize(p)/1e6:.1f} MB)")

print("Writing:")
save(out['M'], "raw_monthly.parquet")
save(out['W'], "raw_weekly.parquet")
save(out['A'], "raw_annual.parquet")
save(fy,       "fiscal_year_end.parquet")
save(meta,     "security_meta.parquet")
log.to_csv(os.path.join(DATA_DIR, "read_log.csv"), index=False)
print(f"  {'read_log.csv':26s} {len(log):>12,} rows")

## Risk-free rate and market indices

Same file structure, with one series per column.

In [ ]:
for fname, outname in [("RF_M.xlsx", "raw_riskfree.parquet"),
                       ("IDX_W.xlsx", "raw_index.parquet")]:
    p = os.path.join(DATA_DIR, fname)
    if not os.path.exists(p):
        print(f"{fname} not found, skipped")
        continue
    v, d, md, nerr = parse_ds_file(p)
    print(f"{fname}: {len(v):,} rows | series: {sorted(v['symbol'].unique())} | #ERROR {nerr}")
    if len(v):
        print(f"  period: {v['date'].min()} -> {v['date'].max()}")
        v.to_parquet(os.path.join(DATA_DIR, outname), index=False)
        print(f"  written to {outname}")

## Check saved files

In [ ]:
for f in ["raw_monthly.parquet", "raw_weekly.parquet", "raw_annual.parquet",
          "fiscal_year_end.parquet", "security_meta.parquet",
          "raw_riskfree.parquet", "raw_index.parquet"]:
    p = os.path.join(DATA_DIR, f)
    if os.path.exists(p):
        d = pd.read_parquet(p)
        print(f"{f:26s} OK  {len(d):>12,} rows x {d.shape[1]} cols")
    else:
        print(f"{f:26s} not present")